# K-Nearest Neighbors and Distance Metric Comparison Across Healthcare Datasets

This notebook applies K-Nearest Neighbors (KNN) models across three healthcare-related datasets: diabetes, life expectancy, and SUPPORT2. The workflow compares several distance metrics, including Euclidean, Manhattan, Minkowski, and cosine distance.

The notebook includes preprocessing with feature scaling, train-test splitting, baseline KNN modeling, distance-metric comparisons, and hyperparameter tuning using GridSearchCV.

This notebook is included in the modeling folder because the main focus is model development, model comparison, distance-metric evaluation, and performance tuning rather than exploratory data analysis.


## Diabetes Dataset: KNN Classification

This section applies KNN classification to the diabetes health indicators dataset. The goal is to compare distance metrics and tuning choices for predicting diabetes outcome classes while considering class imbalance in the target variable.


In [5]:
#Diabetes imputed

import pandas as pd

diabetes = pd.read_csv("diabetes_no_duplicates.csv")
print(diabetes.shape)
diabetes.head()

(229781, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [6]:
from sklearn.model_selection import train_test_split

# Features (all columns except target)
X = diabetes.drop("Diabetes_012", axis=1)

# Target
y = diabetes["Diabetes_012"]

# Train–test split (stratify for class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((183824, 21), (45957, 21))

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Baseline KNN 
knn_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

knn_baseline.fit(X_train, y_train)

y_pred_base = knn_baseline.predict(X_test)
print("Baseline KNN accuracy:", accuracy_score(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base))

Baseline KNN accuracy: 0.8152838522967122
              precision    recall  f1-score   support

         0.0       0.85      0.95      0.90     38012
         1.0       0.05      0.00      0.00       926
         2.0       0.41      0.21      0.27      7019

    accuracy                           0.82     45957
   macro avg       0.44      0.39      0.39     45957
weighted avg       0.77      0.82      0.78     45957



In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

metrics_to_try = {
    "euclidean":  {"metric": "euclidean"},
    "manhattan":  {"metric": "manhattan"},
    "minkowski_p3": {"metric": "minkowski", "p": 3},
    "cosine":     {"metric": "cosine", "n_jobs": -1, "algorithm": "brute"}  # cosine needs brute
}

results = []

for name, params in metrics_to_try.items():
    print(f"\n KNN with {name} distance")
    
    knn = KNeighborsClassifier(
        n_neighbors=5,
        **params
    )
    
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", knn)
    ])
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    print("Accuracy:", acc)
    print(classification_report(y_test, y_pred))
    
    results.append((name, acc))

print("\nSummary of accuracies:")
for name, acc in results:
    print(f"{name}: {acc:.4f}")


 KNN with euclidean distance
Accuracy: 0.8152838522967122
              precision    recall  f1-score   support

         0.0       0.85      0.95      0.90     38012
         1.0       0.05      0.00      0.00       926
         2.0       0.41      0.21      0.27      7019

    accuracy                           0.82     45957
   macro avg       0.44      0.39      0.39     45957
weighted avg       0.77      0.82      0.78     45957


 KNN with manhattan distance
Accuracy: 0.8161324716582893
              precision    recall  f1-score   support

         0.0       0.85      0.95      0.90     38012
         1.0       0.05      0.00      0.00       926
         2.0       0.41      0.21      0.28      7019

    accuracy                           0.82     45957
   macro avg       0.44      0.39      0.39     45957
weighted avg       0.77      0.82      0.78     45957


 KNN with minkowski_p3 distance
Accuracy: 0.8155232064756185
              precision    recall  f1-score   support

   

In [9]:
#Target distribution (class imbalance)
print(diabetes["Diabetes_012"].value_counts())
print(diabetes["Diabetes_012"].value_counts(normalize=True))

#Basic summary of predictors
diabetes.describe()

#Check missing values
diabetes.isna().sum()

Diabetes_012
0.0    190055
2.0     35097
1.0      4629
Name: count, dtype: int64
Diabetes_012
0.0    0.827114
2.0    0.152741
1.0    0.020145
Name: proportion, dtype: float64


Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [10]:
# How many duplicate rows?
diabetes.duplicated().sum()

np.int64(0)

In [11]:
diabetes.shape  # rows, columns

(229781, 22)

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

# Example: final KNN with Manhattan distance
final_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=5,
        metric="manhattan"
    ))
])

final_knn.fit(X_train, y_train)

y_train_pred = final_knn.predict(X_train)
y_test_pred  = final_knn.predict(X_test)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test  accuracy:", accuracy_score(y_test,  y_test_pred))
print("Train macro-F1:", f1_score(y_train, y_train_pred, average="macro"))
print("Test  macro-F1:", f1_score(y_test,  y_test_pred,  average="macro"))

Train accuracy: 0.8568358865001305
Test  accuracy: 0.8161324716582893
Train macro-F1: 0.47447715145027397
Test  macro-F1: 0.39235975784393223


In [13]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, accuracy_score

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="manhattan"))
])

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV macro-F1:", grid.best_score_)

best_knn = grid.best_estimator_

y_test_pred = best_knn.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Test macro-F1:", f1_score(y_test, y_test_pred, average="macro"))

Best params: {'knn__n_neighbors': 3}
Best CV macro-F1: 0.39392511045982115
Test accuracy: 0.803033270230868
Test macro-F1: 0.39478627946312733


In [14]:
#manhattan
from sklearn.metrics import classification_report

print("\nClassification report (test):\n")
print(classification_report(y_test, y_test_pred))


Classification report (test):

              precision    recall  f1-score   support

         0.0       0.85      0.93      0.89     38012
         1.0       0.05      0.00      0.01       926
         2.0       0.37      0.24      0.29      7019

    accuracy                           0.80     45957
   macro avg       0.42      0.39      0.39     45957
weighted avg       0.76      0.80      0.78     45957



In [15]:
param_grid = [
    # Euclidean
    {
        "knn__metric": ["euclidean"],
        "knn__n_neighbors": [5, 9, 13, 17, 21],
        "knn__weights": ["uniform", "distance"],
    },
]

knn_metrics_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

grid_knn = GridSearchCV(
    knn_metrics_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_train, y_train)

print("Best parameters:", grid_knn.best_params_)
print("Best CV accuracy:", grid_knn.best_score_)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test  accuracy:", accuracy_score(y_test,  y_test_pred))
print("Train macro-F1:", f1_score(y_train, y_train_pred, average="macro"))
print("Test  macro-F1:", f1_score(y_test,  y_test_pred,  average="macro"))

print("\nClassification report (test):\n")
print(classification_report(y_test, y_test_pred))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'knn__metric': 'euclidean', 'knn__n_neighbors': 21, 'knn__weights': 'uniform'}
Best CV accuracy: 0.8290158023159708
Train accuracy: 0.8568358865001305
Test  accuracy: 0.803033270230868
Train macro-F1: 0.47447715145027397
Test  macro-F1: 0.39478627946312733

Classification report (test):

              precision    recall  f1-score   support

         0.0       0.85      0.93      0.89     38012
         1.0       0.05      0.00      0.01       926
         2.0       0.37      0.24      0.29      7019

    accuracy                           0.80     45957
   macro avg       0.42      0.39      0.39     45957
weighted avg       0.76      0.80      0.78     45957



In [16]:
print(grid_knn.best_params_)
print(grid_knn.best_score_)

{'knn__metric': 'euclidean', 'knn__n_neighbors': 21, 'knn__weights': 'uniform'}
0.8290158023159708


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

knn_minkowski = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=21,
        metric="minkowski",
        p=3,
        weights="uniform"
    ))
])


knn_minkowski.fit(X_train, y_train)

y_test_pred = knn_minkowski.predict(X_test)

print("Minkowski (p=3, k=21, uniform)")
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Test macro-F1:", f1_score(y_test, y_test_pred, average="macro"))

print("\nClassification report (test):\n")
print(classification_report(y_test, y_test_pred))

Minkowski (p=3, k=21, uniform)
Test accuracy: 0.8295145462062362
Test macro-F1: 0.37097851241339524

Classification report (test):

              precision    recall  f1-score   support

         0.0       0.84      0.98      0.91     38012
         1.0       0.00      0.00      0.00       926
         2.0       0.51      0.13      0.21      7019

    accuracy                           0.83     45957
   macro avg       0.45      0.37      0.37     45957
weighted avg       0.77      0.83      0.78     45957



In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# KNN with cosine distance, using k=21 and uniform weights
knn_cosine = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=21,
        metric="cosine",
        weights="uniform"
    ))
])

knn_cosine.fit(X_train, y_train)

# Only predict on test to avoid extra cost
y_test_pred = knn_cosine.predict(X_test)

print("Cosine (k=21, uniform)")
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Test macro-F1:", f1_score(y_test, y_test_pred, average="macro"))

print("\nClassification report (test):\n")
print(classification_report(y_test, y_test_pred))

Cosine (k=21, uniform)
Test accuracy: 0.8303414060970037
Test macro-F1: 0.37785752502867265

Classification report (test):

              precision    recall  f1-score   support

         0.0       0.84      0.98      0.91     38012
         1.0       0.00      0.00      0.00       926
         2.0       0.52      0.15      0.23      7019

    accuracy                           0.83     45957
   macro avg       0.45      0.37      0.38     45957
weighted avg       0.78      0.83      0.78     45957



### Diabetes Modeling Summary

The diabetes KNN models achieved relatively high overall accuracy, but minority-class performance remained weak, especially for the prediabetes class. This suggests that distance-metric tuning alone was not enough to address the class imbalance problem in this dataset.


## Life Expectancy Dataset: KNN Regression

This section applies KNN regression to the life expectancy dataset. Euclidean, Manhattan, Minkowski, and cosine distance metrics are compared using RMSE, MAE, and R² to evaluate predictive performance.


In [2]:
#Life Expectancy imputed

import pandas as pd

life_imputed = pd.read_csv("Life Expectancy Data - mean_mode_imputed.csv")

print(life_imputed.shape)
life_imputed.head()

(2938, 22)


,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.0,8.16,65.0,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,58.0,8.18,62.0,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,62.0,8.13,64.0,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272.0,69,0.01,78.184215,67.0,2787,...,67.0,8.52,67.0,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275.0,71,0.01,7.097109,68.0,3013,...,68.0,7.87,68.0,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [4]:
#setup train and test split

from sklearn.model_selection import train_test_split

# Drop categorical columns to keep numeric features only for KNN
life_num = life_imputed.drop(columns=["Country", "Status"])

target_col = "Life expectancy "

X_le = life_num.drop(columns=[target_col])
y_le = life_num[target_col]

X_le_train, X_le_test, y_le_train, y_le_test = train_test_split(
    X_le, y_le, test_size=0.2, random_state=42
)

print(X_le_train.shape, X_le_test.shape)

(2350, 19) (588, 19)


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Baseline KNN (k=5, Euclidean)
knn_le_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))  # default metric = Euclidean
])

knn_le_baseline.fit(X_le_train, y_le_train)

y_le_pred_base = knn_le_baseline.predict(X_le_test)

rmse_base = np.sqrt(mean_squared_error(y_le_test, y_le_pred_base))
mae_base  = mean_absolute_error(y_le_test, y_le_pred_base)
r2_base   = r2_score(y_le_test, y_le_pred_base)

print("Baseline KNN (k=5, Euclidean)")
print("Test RMSE:", rmse_base)
print("Test MAE :", mae_base)
print("Test R^2  :", r2_base)

Baseline KNN (k=5, Euclidean)
Test RMSE: 2.5344649293638417
Test MAE : 1.7869560146462953
Test R^2  : 0.9258556940374291


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

metrics_to_try = {
    "euclidean": {"metric": "euclidean"},
    "manhattan": {"metric": "manhattan"},
    "minkowski_p3": {"metric": "minkowski", "p": 3},
    "cosine": {"metric": "cosine", "algorithm": "brute"},  # cosine needs brute
}

results = []

for name, params in metrics_to_try.items():
    print(f"\nKNN with {name} distance")

    knn = KNeighborsRegressor(
        n_neighbors=5,
        **params
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", knn),
    ])

    model.fit(X_le_train, y_le_train)
    y_pred = model.predict(X_le_test)

    rmse = np.sqrt(mean_squared_error(y_le_test, y_pred))
    mae  = mean_absolute_error(y_le_test, y_pred)
    r2   = r2_score(y_le_test, y_pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R²  : {r2:.4f}")

    results.append((name, rmse, mae, r2))

print("\nSummary of metrics (k=5):")
for name, rmse, mae, r2 in results:
    print(f"{name:12s}  RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")


KNN with euclidean distance
RMSE: 2.5345
MAE : 1.7870
R²  : 0.9259

KNN with manhattan distance
RMSE: 2.0457
MAE : 1.4059
R²  : 0.9517

KNN with minkowski_p3 distance
RMSE: 2.9598
MAE : 2.0715
R²  : 0.8989

KNN with cosine distance
RMSE: 2.7190
MAE : 1.9399
R²  : 0.9147

Summary of metrics (k=5):
euclidean     RMSE=2.5345, MAE=1.7870, R²=0.9259
manhattan     RMSE=2.0457, MAE=1.4059, R²=0.9517
minkowski_p3  RMSE=2.9598, MAE=2.0715, R²=0.8989
cosine        RMSE=2.7190, MAE=1.9399, R²=0.9147


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Manhattan KNN pipeline
knn_le_manhattan = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(metric="manhattan"))
])

# Tune k and weights around the good region
param_grid_manhattan = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_le_manhattan = GridSearchCV(
    knn_le_manhattan,
    param_grid_manhattan,
    cv=5,
    scoring="neg_root_mean_squared_error",  # lower RMSE is better
    n_jobs=1,
    verbose=1
)

grid_le_manhattan.fit(X_le_train, y_le_train)

best_manhattan = grid_le_manhattan.best_estimator_
y_le_test_pred_manhattan = best_manhattan.predict(X_le_test)

rmse_manhattan = np.sqrt(mean_squared_error(y_le_test, y_le_test_pred_manhattan))
mae_manhattan  = mean_absolute_error(y_le_test, y_le_test_pred_manhattan)
r2_manhattan   = r2_score(y_le_test, y_le_test_pred_manhattan)

print("Manhattan KNN – best params:", grid_le_manhattan.best_params_)
print("Manhattan KNN – best CV RMSE:", -grid_le_manhattan.best_score_)

print("\nManhattan KNN – test performance")
print("Test RMSE:", rmse_manhattan)
print("Test MAE :", mae_manhattan)
print("Test R²  :", r2_manhattan)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Manhattan KNN – best params: {'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Manhattan KNN – best CV RMSE: 2.3147597941630056

Manhattan KNN – test performance
Test RMSE: 1.9961531469384208
Test MAE : 1.2302225263980398
Test R²  : 0.9540068719317358


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Euclidean KNN pipeline
knn_le_euclidean = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(metric="euclidean"))
])

param_grid_euclidean = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_le_euclidean = GridSearchCV(
    knn_le_euclidean,
    param_grid_euclidean,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    verbose=1
)

grid_le_euclidean.fit(X_le_train, y_le_train)

best_euclidean = grid_le_euclidean.best_estimator_
y_le_test_pred_euc = best_euclidean.predict(X_le_test)

rmse_euc = np.sqrt(mean_squared_error(y_le_test, y_le_test_pred_euc))
mae_euc  = mean_absolute_error(y_le_test, y_le_test_pred_euc)
r2_euc   = r2_score(y_le_test, y_le_test_pred_euc)

print("Euclidean KNN – best params:", grid_le_euclidean.best_params_)
print("Euclidean KNN – best CV RMSE:", -grid_le_euclidean.best_score_)

print("\nEuclidean KNN – test performance")
print("Test RMSE:", rmse_euc)
print("Test MAE :", mae_euc)
print("Test R²  :", r2_euc)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Euclidean KNN – best params: {'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Euclidean KNN – best CV RMSE: 2.885972472305336

Euclidean KNN – test performance
Test RMSE: 2.426912823857328
Test MAE : 1.6116283214214764
Test R²  : 0.932014924348915


In [11]:
# Minkowski (p = 3) KNN pipeline
knn_le_minkowski = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(metric="minkowski", p=3))
])

param_grid_minkowski = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_le_minkowski = GridSearchCV(
    knn_le_minkowski,
    param_grid_minkowski,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    verbose=1
)

grid_le_minkowski.fit(X_le_train, y_le_train)

best_minkowski = grid_le_minkowski.best_estimator_
y_le_test_pred_mink = best_minkowski.predict(X_le_test)

rmse_mink = np.sqrt(mean_squared_error(y_le_test, y_le_test_pred_mink))
mae_mink  = mean_absolute_error(y_le_test, y_le_test_pred_mink)
r2_mink   = r2_score(y_le_test, y_le_test_pred_mink)

print("Minkowski(p=3) KNN – best params:", grid_le_minkowski.best_params_)
print("Minkowski(p=3) KNN – best CV RMSE:", -grid_le_minkowski.best_score_)

print("\nMinkowski(p=3) KNN – test performance")
print("Test RMSE:", rmse_mink)
print("Test MAE :", mae_mink)
print("Test R²  :", r2_mink)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Minkowski(p=3) KNN – best params: {'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Minkowski(p=3) KNN – best CV RMSE: 3.1450354261600646

Minkowski(p=3) KNN – test performance
Test RMSE: 2.7095994630858278
Test MAE : 1.8695624076399304
Test R²  : 0.9152547378800285


In [12]:
# Cosine KNN pipeline
knn_le_cosine = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(metric="cosine", algorithm="brute"))
])

param_grid_cosine = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_le_cosine = GridSearchCV(
    knn_le_cosine,
    param_grid_cosine,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
    verbose=1
)

grid_le_cosine.fit(X_le_train, y_le_train)

best_cosine = grid_le_cosine.best_estimator_
y_le_test_pred_cos = best_cosine.predict(X_le_test)

rmse_cos = np.sqrt(mean_squared_error(y_le_test, y_le_test_pred_cos))
mae_cos  = mean_absolute_error(y_le_test, y_le_test_pred_cos)
r2_cos   = r2_score(y_le_test, y_le_test_pred_cos)

print("Cosine KNN – best params:", grid_le_cosine.best_params_)
print("Cosine KNN – best CV RMSE:", -grid_le_cosine.best_score_)

print("\nCosine KNN – test performance")
print("Test RMSE:", rmse_cos)
print("Test MAE :", mae_cos)
print("Test R²  :", r2_cos)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Cosine KNN – best params: {'knn__n_neighbors': 5, 'knn__weights': 'distance'}
Cosine KNN – best CV RMSE: 2.879020972723281

Cosine KNN – test performance
Test RMSE: 2.510449409596627
Test MAE : 1.6532222788884874
Test R²  : 0.9272541571618754


### Life Expectancy Modeling Summary

The life expectancy models performed well overall. Among the tested distance metrics, Manhattan distance with tuning produced the strongest results, showing that distance-metric selection can meaningfully affect KNN regression performance.


## SUPPORT2 Dataset: KNN Classification

This section applies KNN classification to the SUPPORT2 dataset using hospital death as the target outcome. Multiple distance metrics and tuning strategies are compared using accuracy, macro-F1, and classification reports.


In [16]:
#Support2 imputed
import pandas as pd

support2_imputed = pd.read_csv("support2_imputed.csv")

print(support2_imputed.shape)
support2_imputed.head()

(9105, 45)


,age,sex,dzgroup,dzclass,num.co,edu,income,scoma,charges,totcst,...,ph,glucose,bun,urine,adlp,adls,adlsc,death,hospdead,sfdm2
0,62.84998,male,Lung Cancer,Cancer,0,11.000000,$11-$25k,0.0,9715.0,30825.867768,...,7.459961,159.873398,32.349463,2191.546047,7.00000,7.0,7.0,0,0,<2 mo. follow-up
1,60.33899,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.000000,$11-$25k,44.0,34496.0,30825.867768,...,7.250000,159.873398,32.349463,2191.546047,1.15791,1.0,1.0,1,1,<2 mo. follow-up
2,52.74698,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.000000,under $11k,0.0,41094.0,30825.867768,...,7.459961,159.873398,32.349463,2191.546047,1.00000,0.0,0.0,1,0,<2 mo. follow-up
3,42.38498,female,Lung Cancer,Cancer,2,11.000000,under $11k,0.0,3075.0,30825.867768,...,7.415364,159.873398,32.349463,2191.546047,0.00000,0.0,0.0,1,0,no(M2 and SIP pres)
4,79.88495,female,ARF/MOSF w/Sepsis,ARF/MOSF,1,11.747691,under $11k,26.0,50127.0,30825.867768,...,7.509766,159.873398,32.349463,2191.546047,1.15791,2.0,2.0,0,0,no(M2 and SIP pres)


In [17]:
from sklearn.model_selection import train_test_split

target_col = "hospdead"   # keep this as the target

# Make y 
y_sup = support2_imputed[target_col]

#X using only numeric columns
X_sup = support2_imputed.drop(columns=[target_col]).select_dtypes(include=["number"])

print("X_sup columns:", X_sup.columns)
print(X_sup.dtypes.head())

#Train/test split
X_sup_train, X_sup_test, y_sup_train, y_sup_test = train_test_split(
    X_sup, y_sup, test_size=0.2, random_state=42, stratify=y_sup
)

print(X_sup_train.shape, X_sup_test.shape)

X_sup columns: Index(['age', 'num.co', 'edu', 'scoma', 'charges', 'totcst', 'totmcst',
       'avtisst', 'sps', 'aps', 'surv2m', 'surv6m', 'hday', 'diabetes',
       'dementia', 'prg2m', 'prg6m', 'dnrday', 'meanbp', 'wblc', 'hrt', 'resp',
       'temp', 'pafi', 'alb', 'bili', 'crea', 'sod', 'ph', 'glucose', 'bun',
       'urine', 'adlp', 'adls', 'adlsc', 'death'],
      dtype='object')
age        float64
num.co       int64
edu        float64
scoma      float64
charges    float64
dtype: object
(7284, 36) (1821, 36)


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

knn_sup_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))  # default metric = Euclidean
])

knn_sup_baseline.fit(X_sup_train, y_sup_train)

y_sup_test_pred_base = knn_sup_baseline.predict(X_sup_test)

print("Baseline KNN (k=5, Euclidean)")
print("Test accuracy:", accuracy_score(y_sup_test, y_sup_test_pred_base))
print("Test macro-F1:", f1_score(y_sup_test, y_sup_test_pred_base, average="macro"))
print("\nClassification report (test):\n")
print(classification_report(y_sup_test, y_sup_test_pred_base))

Baseline KNN (k=5, Euclidean)
Test accuracy: 0.8605161998901703
Test macro-F1: 0.8066749703241771

Classification report (test):

              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1349
           1       0.78      0.64      0.70       472

    accuracy                           0.86      1821
   macro avg       0.83      0.79      0.81      1821
weighted avg       0.86      0.86      0.86      1821



In [19]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

metrics_to_try = {
    "euclidean":   {"metric": "euclidean"},
    "manhattan":   {"metric": "manhattan"},
    "minkowski_p3": {"metric": "minkowski", "p": 3},
    "cosine":      {"metric": "cosine", "algorithm": "brute"},  # cosine needs brute
}

results_sup = []

for name, params in metrics_to_try.items():
    print(f"\nKNN with {name} distance (k=5)")

    knn = KNeighborsClassifier(
        n_neighbors=5,
        **params
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", knn),
    ])

    model.fit(X_sup_train, y_sup_train)
    y_pred = model.predict(X_sup_test)

    acc = accuracy_score(y_sup_test, y_pred)
    macro_f1 = f1_score(y_sup_test, y_pred, average="macro")

    print(f"Test accuracy: {acc:.4f}")
    print(f"Test macro-F1: {macro_f1:.4f}")
    print("\nClassification report (test):\n")
    print(classification_report(y_sup_test, y_pred))

    results_sup.append((name, acc, macro_f1))

print("\nSummary of metrics at k=5:")
for name, acc, macro_f1 in results_sup:
    print(f"{name:12s}  acc={acc:.4f}, macro-F1={macro_f1:.4f}")


KNN with euclidean distance (k=5)
Test accuracy: 0.8605
Test macro-F1: 0.8067

Classification report (test):

              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1349
           1       0.78      0.64      0.70       472

    accuracy                           0.86      1821
   macro avg       0.83      0.79      0.81      1821
weighted avg       0.86      0.86      0.86      1821


KNN with manhattan distance (k=5)
Test accuracy: 0.8677
Test macro-F1: 0.8199

Classification report (test):

              precision    recall  f1-score   support

           0       0.89      0.93      0.91      1349
           1       0.78      0.68      0.73       472

    accuracy                           0.87      1821
   macro avg       0.84      0.81      0.82      1821
weighted avg       0.86      0.87      0.86      1821


KNN with minkowski_p3 distance (k=5)
Test accuracy: 0.8622
Test macro-F1: 0.8069

Classification report (test):

          

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Pipeline for cosine KNN
knn_sup_cosine = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="cosine", algorithm="brute"))
])

# Small grid over k and weights
param_grid_cosine = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
}

grid_sup_cosine = GridSearchCV(
    knn_sup_cosine,
    param_grid_cosine,
    cv=5,
    scoring="f1_macro",   # focus on macro-F1
    n_jobs=1,
    verbose=1
)

grid_sup_cosine.fit(X_sup_train, y_sup_train)

best_cosine = grid_sup_cosine.best_estimator_
y_sup_test_pred_best = best_cosine.predict(X_sup_test)

print("Cosine KNN – best params:", grid_sup_cosine.best_params_)
print("Cosine KNN – best CV macro-F1:", grid_sup_cosine.best_score_)

print("\nCosine KNN – test performance")
print("Test accuracy:", accuracy_score(y_sup_test, y_sup_test_pred_best))
print("Test macro-F1:", f1_score(y_sup_test, y_sup_test_pred_best, average="macro"))
print("\nClassification report (test):\n")
print(classification_report(y_sup_test, y_sup_test_pred_best))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Cosine KNN – best params: {'knn__n_neighbors': 9, 'knn__weights': 'distance'}
Cosine KNN – best CV macro-F1: 0.8224024464746691

Cosine KNN – test performance
Test accuracy: 0.8747940691927513
Test macro-F1: 0.8337263357543564

Classification report (test):

              precision    recall  f1-score   support

           0       0.91      0.93      0.92      1349
           1       0.77      0.73      0.75       472

    accuracy                           0.87      1821
   macro avg       0.84      0.83      0.83      1821
weighted avg       0.87      0.87      0.87      1821



In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Euclidean KNN pipeline
knn_sup_euclidean = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="euclidean"))
])

param_grid_euclidean = {
    "knn__n_neighbors": [7, 9, 11, 13, 15],
    "knn__weights": ["uniform", "distance"],
}

grid_sup_euclidean = GridSearchCV(
    knn_sup_euclidean,
    param_grid_euclidean,
    cv=5,
    scoring="f1_macro",   # same as cosine
    n_jobs=1,
    verbose=1
)

grid_sup_euclidean.fit(X_sup_train, y_sup_train)

best_euclidean = grid_sup_euclidean.best_estimator_
y_sup_test_pred_euc = best_euclidean.predict(X_sup_test)

print("Euclidean KNN – best params:", grid_sup_euclidean.best_params_)
print("Euclidean KNN – best CV macro-F1:", grid_sup_euclidean.best_score_)

print("\nEuclidean KNN – test performance")
print("Test accuracy:", accuracy_score(y_sup_test, y_sup_test_pred_euc))
print("Test macro-F1:", f1_score(y_sup_test, y_sup_test_pred_euc, average="macro"))
print("\nClassification report (test):\n")
print(classification_report(y_sup_test, y_sup_test_pred_euc))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Euclidean KNN – best params: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Euclidean KNN – best CV macro-F1: 0.8121742561663681

Euclidean KNN – test performance
Test accuracy: 0.8780889621087314
Test macro-F1: 0.8288122186124822

Classification report (test):

              precision    recall  f1-score   support

           0       0.89      0.95      0.92      1349
           1       0.84      0.66      0.74       472

    accuracy                           0.88      1821
   macro avg       0.86      0.81      0.83      1821
weighted avg       0.88      0.88      0.87      1821



In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Manhattan KNN pipeline
knn_sup_manhattan = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="manhattan"))
])

#range of k and weights
param_grid_manhattan = {
    "knn__n_neighbors": [5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
}

grid_sup_manhattan = GridSearchCV(
    knn_sup_manhattan,
    param_grid_manhattan,
    cv=5,
    scoring="f1_macro",   # same as Euclidean
    n_jobs=1,
    verbose=1
)

grid_sup_manhattan.fit(X_sup_train, y_sup_train)

best_man = grid_sup_manhattan.best_estimator_
y_sup_test_pred_man = best_man.predict(X_sup_test)

print("Manhattan KNN – best params:", grid_sup_manhattan.best_params_)
print("Manhattan KNN – best CV macro-F1:", grid_sup_manhattan.best_score_)

print("\nManhattan KNN – test performance")
print("Test accuracy:", accuracy_score(y_sup_test, y_sup_test_pred_man))
print("Test macro-F1:", f1_score(y_sup_test, y_sup_test_pred_man, average="macro"))
print("\nClassification report (test):\n")
print(classification_report(y_sup_test, y_sup_test_pred_man))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Manhattan KNN – best params: {'knn__n_neighbors': 9, 'knn__weights': 'uniform'}
Manhattan KNN – best CV macro-F1: 0.8138727353521389

Manhattan KNN – test performance
Test accuracy: 0.8791872597473915
Test macro-F1: 0.8336179600570139

Classification report (test):

              precision    recall  f1-score   support

           0       0.90      0.95      0.92      1349
           1       0.82      0.69      0.75       472

    accuracy                           0.88      1821
   macro avg       0.86      0.82      0.83      1821
weighted avg       0.88      0.88      0.88      1821



In [26]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Minkowski (p=3) KNN pipeline
knn_sup_minkowski = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(metric="minkowski", p=3))
])

param_grid_minkowski = {
    "knn__n_neighbors": [5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
}

grid_sup_minkowski = GridSearchCV(
    knn_sup_minkowski,
    param_grid_minkowski,
    cv=5,
    scoring="f1_macro",
    n_jobs=1,
    verbose=1
)

grid_sup_minkowski.fit(X_sup_train, y_sup_train)

best_mink = grid_sup_minkowski.best_estimator_
y_sup_test_pred_mink = best_mink.predict(X_sup_test)

print("Minkowski(p=3) KNN – best params:", grid_sup_minkowski.best_params_)
print("Minkowski(p=3) KNN – best CV macro-F1:", grid_sup_minkowski.best_score_)

print("\nMinkowski(p=3) KNN – test performance")
print("Test accuracy:", accuracy_score(y_sup_test, y_sup_test_pred_mink))
print("Test macro-F1:", f1_score(y_sup_test, y_sup_test_pred_mink, average="macro"))
print("\nClassification report (test):\n")
print(classification_report(y_sup_test, y_sup_test_pred_mink))

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Minkowski(p=3) KNN – best params: {'knn__n_neighbors': 15, 'knn__weights': 'distance'}
Minkowski(p=3) KNN – best CV macro-F1: 0.8034389286390905

Minkowski(p=3) KNN – test performance
Test accuracy: 0.8682042833607908
Test macro-F1: 0.8121079132731674

Classification report (test):

              precision    recall  f1-score   support

           0       0.88      0.95      0.91      1349
           1       0.83      0.62      0.71       472

    accuracy                           0.87      1821
   macro avg       0.85      0.79      0.81      1821
weighted avg       0.86      0.87      0.86      1821



### SUPPORT2 Modeling Summary

The SUPPORT2 KNN models produced strong classification performance, with tuned Manhattan and cosine distance approaches performing especially well. These results show that KNN can be useful for this dataset when scaling, distance metrics, and hyperparameters are carefully selected.
